# Notebook 02: Model Monitoring Dashboard

**Module 08 — Production Systems**  
**Estimated time**: 12 minutes

## Learning Objectives

By the end of this notebook you will:

1. Compute rolling RMSE, MAE, and bias over an 8-quarter window
2. Run the bias t-test and interpret the result
3. Implement the CUSUM test and plot the stability bands
4. Operate the `ReEstimationTrigger` and observe when it fires
5. Generate and interpret the model comparison table

## Prerequisites

- Module 08 Notebook 01 (pipeline end-to-end)
- Module 08 Guide 02 (monitoring and reporting)

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from scipy import stats
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Setup complete')

In [ ]:
learning_objectives(["Compute rolling RMSE, MAE, and bias over an 8-quarter window", "Run the bias t-test and interpret the result", "Implement the CUSUM test and plot the stability bands", "Operate the `ReEstimationTrigger` and observe when it fires", "Generate and interpret the model comparison table"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Part 1: Synthetic Evaluation History

We simulate 24 quarters of nowcasting history for three models:
- **ElasticNet MIDAS**: good accuracy, moderate stability
- **Ridge MIDAS**: slightly worse than ElasticNet but very stable
- **Equal-weight combination**: best overall accuracy

We also inject a structural break at quarter 16 (approximately 4 years in) to demonstrate the CUSUM test.

In [ ]:
def simulate_forecast_history(
    n_quarters: int = 24,
    break_at: int = 16,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Simulate 24 quarters of nowcasting errors for three models.

    A structural break occurs at `break_at`: the ElasticNet model
    becomes biased post-break (simulates a regime change the model
    does not immediately adapt to).

    Returns DataFrame with columns:
        quarter, actual, en_forecast, ridge_forecast, combo_forecast
    """
    rng = np.random.default_rng(seed)

    # True GDP growth: AR1 process with structural break in volatility
    actual = np.zeros(n_quarters)
    actual[0] = rng.normal(0.5, 0.5)
    for t in range(1, n_quarters):
        # Post-break: higher volatility + lower mean
        if t >= break_at:
            actual[t] = 0.2 * actual[t-1] + rng.normal(-0.3, 0.8)
        else:
            actual[t] = 0.4 * actual[t-1] + rng.normal(0.3, 0.4)

    # ElasticNet: good pre-break, biased post-break
    en_error = np.zeros(n_quarters)
    for t in range(n_quarters):
        if t >= break_at:
            # Systematic overestimation post-break
            en_error[t] = rng.normal(0.35, 0.5)
        else:
            en_error[t] = rng.normal(0.0, 0.35)

    en_forecast = actual + en_error

    # Ridge: less accurate than EN pre-break but less affected by break
    ridge_error = rng.normal(0.05, 0.45, n_quarters)
    # Post-break: moderate positive bias
    ridge_error[break_at:] += 0.15
    ridge_forecast = actual + ridge_error

    # Equal-weight combination: reduces both biases
    combo_forecast = 0.5 * en_forecast + 0.5 * ridge_forecast

    quarters = pd.date_range('2018-01-01', periods=n_quarters, freq='QS')

    return pd.DataFrame({
        'quarter': quarters,
        'actual': actual,
        'en_forecast': en_forecast,
        'ridge_forecast': ridge_forecast,
        'combo_forecast': combo_forecast,
        'en_error': en_error,
        'ridge_error': ridge_error,
        'combo_error': combo_forecast - actual,
        'post_break': np.arange(n_quarters) >= break_at,
    })


df = simulate_forecast_history(n_quarters=24, break_at=16)
print(f'Simulated {len(df)} quarters of forecast history')
print(f'Break at quarter 16 ({df.loc[16, "quarter"].date()})')
print()
print('Pre-break EN bias:', df[~df['post_break']]['en_error'].mean().round(4))
print('Post-break EN bias:', df[df['post_break']]['en_error'].mean().round(4))

## Part 2: Rolling Accuracy Metrics

We compute the rolling 8-quarter RMSE, MAE, and bias for the ElasticNet model. The chart shows how these metrics evolve over the evaluation period, with the structural break visible as a sudden deterioration.

In [ ]:
def compute_rolling_metrics(
    errors: np.ndarray,
    window: int = 8,
) -> pd.DataFrame:
    """Compute rolling RMSE, MAE, and bias."""
    n = len(errors)
    rmse, mae, bias = [], [], []
    for i in range(n):
        start = max(0, i - window + 1)
        w = errors[start : i + 1]
        rmse.append(float(np.sqrt(np.mean(w ** 2))))
        mae.append(float(np.mean(np.abs(w))))
        bias.append(float(np.mean(w)))
    return pd.DataFrame({'rmse': rmse, 'mae': mae, 'bias': bias})


en_metrics = compute_rolling_metrics(df['en_error'].values, window=8)
ridge_metrics = compute_rolling_metrics(df['ridge_error'].values, window=8)
combo_metrics = compute_rolling_metrics(df['combo_error'].values, window=8)

BACKTEST_RMSE = en_metrics['rmse'].iloc[15]  # RMSE at end of pre-break period
REESTIM_THRESHOLD = BACKTEST_RMSE * 1.20

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
quarters = df['quarter']
break_q = df.loc[16, 'quarter']

for ax, metric, title, ylbl in zip(
    axes,
    ['rmse', 'mae', 'bias'],
    ['Rolling RMSE (8Q window)', 'Rolling MAE (8Q window)', 'Rolling Bias (8Q window)'],
    ['RMSE', 'MAE', 'Bias'],
):
    ax.plot(quarters, en_metrics[metric], color='steelblue', linewidth=1.8,
            marker='o', markersize=4, label='ElasticNet')
    ax.plot(quarters, ridge_metrics[metric], color='darkorange', linewidth=1.5,
            linestyle='--', marker='s', markersize=4, label='Ridge')
    ax.plot(quarters, combo_metrics[metric], color='green', linewidth=1.5,
            linestyle=':', marker='^', markersize=4, label='Combination')

    if metric == 'rmse':
        ax.axhline(BACKTEST_RMSE, color='gray', linewidth=1.2, linestyle='-.',
                   label=f'Baseline ({BACKTEST_RMSE:.3f})')
        ax.axhline(REESTIM_THRESHOLD, color='firebrick', linewidth=1.2, linestyle='--',
                   label=f'Re-est. threshold ({REESTIM_THRESHOLD:.3f})')
    if metric == 'bias':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='-')

    ax.axvline(break_q, color='purple', linewidth=1.2, linestyle=':', alpha=0.7,
               label='Structural break')
    ax.set_ylabel(ylbl)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(loc='upper left', fontsize=7, ncol=3)
    ax.grid(axis='y', alpha=0.3)

axes[-1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-Q%q'))
plt.xticks(rotation=30, ha='right')
plt.suptitle('Rolling Forecast Accuracy — Three Models', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Backtest RMSE (baseline): {BACKTEST_RMSE:.4f}')
print(f'Re-estimation threshold (120%): {REESTIM_THRESHOLD:.4f}')
print(f'Final EN RMSE: {en_metrics["rmse"].iloc[-1]:.4f}')
print(f'Final Combo RMSE: {combo_metrics["rmse"].iloc[-1]:.4f}')

## Part 3: Bias Testing

We test the null hypothesis $H_0: \mathbb{E}[\hat{y} - y] = 0$ for both the pre-break and post-break periods separately.

In [ ]:
def test_forecast_bias(errors: np.ndarray, alpha: float = 0.05) -> Dict:
    """Two-sided t-test for zero mean forecast error."""
    n = len(errors)
    mean_e = float(np.mean(errors))
    se = float(np.std(errors, ddof=1) / np.sqrt(n))
    t_stat = mean_e / se if se > 0 else 0.0
    p_value = float(2 * stats.t.sf(abs(t_stat), df=n - 1))
    return {
        'n': n,
        'mean_error': mean_e,
        'se': se,
        't_stat': t_stat,
        'p_value': p_value,
        'reject': p_value < alpha,
        'ci_95': (mean_e - 1.96 * se, mean_e + 1.96 * se),
    }


pre_break = df[~df['post_break']]['en_error'].values
post_break = df[df['post_break']]['en_error'].values
all_en = df['en_error'].values

for label, errors in [
    ('Pre-break', pre_break),
    ('Post-break', post_break),
    ('Full sample', all_en),
]:
    result = test_forecast_bias(errors)
    status = 'BIAS DETECTED' if result['reject'] else 'No bias'
    print(f'{label} ({result["n"]} obs): '
          f'mean={result["mean_error"]:+.3f}, '
          f'se={result["se"]:.3f}, '
          f't={result["t_stat"]:+.2f}, '
          f'p={result["p_value"]:.4f} → {status}')

## Part 4: CUSUM Structural Break Test

The CUSUM test uses recursive residuals from an AR(1) model fit on the errors sequence. The cumulative sum should stay within the $\pm c_{0.05} \sqrt{n-1}$ bands if the model is stable.

In [ ]:
def cusum_test_simple(
    errors: np.ndarray,
    alpha: float = 0.05,
) -> Dict:
    """
    Simplified CUSUM test on forecast errors.

    Tests whether the mean of the errors is stable over time by
    computing cumulative standardised deviations from zero.

    Returns dict with: cusum, upper, lower, any_crossing, break_index
    """
    n = len(errors)
    c_alpha = {0.01: 1.143, 0.05: 0.948, 0.10: 0.850}.get(alpha, 0.948)

    sigma = np.std(errors, ddof=1)
    standardised = (errors - errors.mean()) / (sigma + 1e-12)
    cusum = np.cumsum(standardised)

    t_arr = np.arange(1, n + 1)
    upper = c_alpha * np.sqrt(n) * (1 + 2 * t_arr / n)
    lower = -upper

    crossings = np.where((cusum > upper) | (cusum < lower))[0]
    any_crossing = len(crossings) > 0
    break_idx = int(crossings[0]) if any_crossing else None

    return {
        'cusum': cusum,
        'upper': upper,
        'lower': lower,
        'any_crossing': any_crossing,
        'break_index': break_idx,
    }


cusum_result = cusum_test_simple(df['en_error'].values)

# Plot
fig, ax = plt.subplots(figsize=(11, 4))
t_idx = np.arange(len(df))

ax.plot(df['quarter'], cusum_result['cusum'], color='steelblue',
        linewidth=2, marker='o', markersize=4, label='CUSUM')
ax.fill_between(
    df['quarter'],
    cusum_result['lower'],
    cusum_result['upper'],
    alpha=0.12, color='gray', label='5% significance band'
)
ax.plot(df['quarter'], cusum_result['upper'], color='gray',
        linewidth=1, linestyle='--')
ax.plot(df['quarter'], cusum_result['lower'], color='gray',
        linewidth=1, linestyle='--')
ax.axhline(0, color='black', linewidth=0.8)

if cusum_result['break_index'] is not None:
    break_q_detected = df['quarter'].iloc[cusum_result['break_index']]
    ax.axvline(break_q_detected, color='firebrick', linewidth=2,
               linestyle=':', label=f'Break detected: {break_q_detected.date()}')

ax.axvline(df.loc[16, 'quarter'], color='purple', linewidth=1.5,
           linestyle=':', alpha=0.6, label='True break (Q16)')

ax.set_title('CUSUM Test — ElasticNet Forecast Errors', fontsize=12, fontweight='bold')
ax.set_xlabel('Quarter')
ax.set_ylabel('CUSUM')
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Structural break detected: {cusum_result["any_crossing"]}')
if cusum_result['break_index'] is not None:
    print(f'Detected at index: {cusum_result["break_index"]} '
          f'(quarter: {df["quarter"].iloc[cusum_result["break_index"]].date()})')
    print(f'True break at index: 16 (quarter: {df.loc[16, "quarter"].date()})')

## Part 5: Re-Estimation Trigger

We simulate the trigger firing over the 24-quarter history and observe which trigger type fires first.

In [ ]:
class ReEstimationTrigger:
    """Three-trigger re-estimation monitor."""

    def __init__(
        self,
        backtest_rmse: float,
        rmse_threshold: float = 0.20,
        calendar_quarters: int = 4,
    ):
        self.backtest_rmse = backtest_rmse
        self.threshold_rmse = backtest_rmse * (1 + rmse_threshold)
        self.calendar_quarters = calendar_quarters
        self._since_last = 0
        self._consecutive_exceed = 0

    def check(self, current_rmse: float, cusum_break: bool) -> Dict:
        self._since_last += 1
        reasons = []

        if self._since_last >= self.calendar_quarters:
            reasons.append(f'calendar ({self._since_last}Q)')

        if current_rmse > self.threshold_rmse:
            self._consecutive_exceed += 1
            if self._consecutive_exceed >= 2:
                reasons.append(
                    f'performance (RMSE {current_rmse:.3f} > '
                    f'{self.threshold_rmse:.3f})'
                )
        else:
            self._consecutive_exceed = 0

        if cusum_break:
            reasons.append('structural break (CUSUM)')

        fired = len(reasons) > 0
        if fired:
            self._since_last = 0

        return {
            'fired': fired,
            'reason': '; '.join(reasons) if reasons else 'none',
        }


trigger = ReEstimationTrigger(
    backtest_rmse=BACKTEST_RMSE,
    rmse_threshold=0.20,
    calendar_quarters=4,
)

cusum_crossed = False
print(f'Monitoring 24 quarters (baseline RMSE = {BACKTEST_RMSE:.4f}, '
      f'threshold = {REESTIM_THRESHOLD:.4f})')
print()

for i, row in df.iterrows():
    rolling_rmse = en_metrics['rmse'].iloc[i]

    # CUSUM break fires on the first crossing
    if (
        cusum_result['break_index'] is not None
        and i == cusum_result['break_index']
    ):
        cusum_crossed = True

    result = trigger.check(rolling_rmse, cusum_crossed)
    cusum_crossed = False  # only fire once

    status = f'TRIGGER: {result["reason"]}' if result['fired'] else '.'
    print(
        f'Q{i+1:02d} {str(row["quarter"].date()):12s} '
        f'RMSE={rolling_rmse:.3f} {status}'
    )

## Part 6: Model Comparison Table

The Diebold-Mariano test compares the forecast accuracy of each model against the ElasticNet baseline.

In [ ]:
def diebold_mariano_test(
    errors_baseline: np.ndarray,
    errors_challenger: np.ndarray,
) -> Dict:
    """
    Diebold-Mariano test: H0 equal MSE between baseline and challenger.
    Uses Newey-West variance with bandwidth h=1.

    Negative DM stat = challenger is better (lower MSE).
    """
    d = errors_baseline ** 2 - errors_challenger ** 2
    n = len(d)
    d_bar = np.mean(d)
    gamma0 = np.var(d, ddof=1)
    gamma1 = float(np.cov(d[:-1], d[1:])[0, 1]) if n > 1 else 0.0
    nw_var = max(gamma0 + 2 * gamma1, 1e-12)
    dm_stat = d_bar / np.sqrt(nw_var / n)
    p_value = float(2 * stats.t.sf(abs(dm_stat), df=n - 1))
    return {
        'dm_stat': float(dm_stat),
        'p_value': p_value,
        'challenger_better': dm_stat > 0 and p_value < 0.10,
    }


en_errors = df['en_error'].values
ridge_errors = df['ridge_error'].values
combo_errors = df['combo_error'].values

# AR1 benchmark: naive last-value forecast
ar1_errors = np.zeros(len(df))
actuals = df['actual'].values
for t in range(1, len(df)):
    ar1_pred = actuals[t - 1]
    ar1_errors[t] = ar1_pred - actuals[t]
ar1_errors[0] = 0.0

model_errors = {
    'ElasticNet': en_errors,
    'Ridge': ridge_errors,
    'Combination': combo_errors,
    'AR(1)': ar1_errors,
}

rows = []
for name, errors in model_errors.items():
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    mae = float(np.mean(np.abs(errors)))
    bias = float(np.mean(errors))

    if name != 'ElasticNet':
        dm = diebold_mariano_test(en_errors, errors)
        dm_stat = dm['dm_stat']
        dm_pval = dm['p_value']
        better = dm['challenger_better']
    else:
        dm_stat = float('nan')
        dm_pval = float('nan')
        better = False

    rows.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'Bias': bias,
        'DM stat vs EN': dm_stat,
        'DM p-value': dm_pval,
        'Better than EN (10%)': better,
    })

comparison = pd.DataFrame(rows).sort_values('RMSE')
pd.set_option('display.float_format', '{:.4f}'.format)
print('\nModel Comparison Table')
print(comparison.to_string(index=False))

best_model = comparison.iloc[0]['Model']
print(f'\nRecommendation: deploy {best_model}')

## Part 7: Health Report Generation

We assemble the daily health report as a structured text output — the same format logged after every pipeline run.

In [ ]:
import datetime


def generate_health_report(
    quarter_idx: int,
    df: pd.DataFrame,
    rolling_metrics: pd.DataFrame,
    bias_result: Dict,
    cusum_result: Dict,
    trigger_result: Dict,
    point_forecast: float,
    interval_95: Tuple[float, float],
) -> str:
    n_train = quarter_idx
    q = df.loc[quarter_idx, 'quarter']
    rm = rolling_metrics.iloc[quarter_idx]

    lines = [
        '=' * 60,
        'NOWCASTING PIPELINE HEALTH REPORT',
        f'Run timestamp : {datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")}',
        f'Forecast date : {q.date()}',
        f'Target period : {q.to_period("Q")}',
        '=' * 60,
        '',
        'CURRENT NOWCAST',
        f'  Point estimate : {point_forecast:+.3f}',
        f'  95% interval   : [{interval_95[0]:+.3f}, {interval_95[1]:+.3f}]',
        f'  Training obs   : {n_train}',
        '',
        'ROLLING ACCURACY (last 8 quarters)',
        f'  RMSE : {rm["rmse"]:.4f}',
        f'  MAE  : {rm["mae"]:.4f}',
        f'  Bias : {rm["bias"]:+.4f}',
        '',
        'BIAS TEST',
        f'  Mean error = {bias_result["mean_error"]:+.3f}, '
        f'p = {bias_result["p_value"]:.4f}',
        f'  {"BIAS DETECTED" if bias_result["reject"] else "No significant bias"}',
        '',
        'CUSUM STABILITY',
        f'  Break detected: {cusum_result["any_crossing"]}',
        '',
        'RE-ESTIMATION',
        f'  Trigger: {trigger_result["reason"]}',
        f'  Action : {"RE-ESTIMATED" if trigger_result["fired"] else "No action"}',
        '',
        '=' * 60,
    ]
    return '\n'.join(lines)


# Generate report for the last quarter in the sample
last_idx = len(df) - 1
last_errors = df['en_error'].values
last_bias = test_forecast_bias(last_errors)
last_cusum = cusum_test_simple(last_errors)
last_trigger = {'fired': False, 'reason': 'none'}

last_point = float(df.loc[last_idx, 'en_forecast'])
last_sigma = float(np.std(last_errors))
last_interval = (last_point - 1.96 * last_sigma, last_point + 1.96 * last_sigma)

report = generate_health_report(
    last_idx,
    df,
    en_metrics,
    last_bias,
    last_cusum,
    last_trigger,
    last_point,
    last_interval,
)
print(report)

## Part 8: Full Dashboard

Combine all monitoring charts into a single two-by-two dashboard figure.

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

# Panel 1: Rolling RMSE
ax1.plot(df['quarter'], en_metrics['rmse'], color='steelblue', linewidth=2,
         marker='o', markersize=4, label='ElasticNet')
ax1.plot(df['quarter'], combo_metrics['rmse'], color='green', linewidth=1.5,
         linestyle=':', marker='^', markersize=4, label='Combination')
ax1.axhline(BACKTEST_RMSE, color='gray', linewidth=1, linestyle='-.', label='Baseline')
ax1.axhline(REESTIM_THRESHOLD, color='firebrick', linewidth=1, linestyle='--', label='Threshold')
ax1.axvline(df.loc[16, 'quarter'], color='purple', linewidth=1, linestyle=':', alpha=0.7)
ax1.set_title('Rolling RMSE (8Q)', fontweight='bold')
ax1.legend(fontsize=7)
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=30)

# Panel 2: CUSUM
ax2.plot(df['quarter'], cusum_result['cusum'], color='steelblue', linewidth=2,
         marker='o', markersize=3, label='CUSUM')
ax2.fill_between(df['quarter'], cusum_result['lower'], cusum_result['upper'],
                 alpha=0.12, color='gray', label='5% band')
ax2.plot(df['quarter'], cusum_result['upper'], color='gray', linewidth=0.8, linestyle='--')
ax2.plot(df['quarter'], cusum_result['lower'], color='gray', linewidth=0.8, linestyle='--')
ax2.axhline(0, color='black', linewidth=0.7)
if cusum_result['break_index'] is not None:
    ax2.axvline(
        df['quarter'].iloc[cusum_result['break_index']],
        color='firebrick', linewidth=1.5, linestyle=':', label='Break detected'
    )
ax2.axvline(df.loc[16, 'quarter'], color='purple', linewidth=1, linestyle=':', alpha=0.7,
            label='True break')
ax2.set_title('CUSUM Stability Test', fontweight='bold')
ax2.legend(fontsize=7)
ax2.grid(axis='y', alpha=0.3)
ax2.tick_params(axis='x', rotation=30)

# Panel 3: Bias over time
ax3.plot(df['quarter'], en_metrics['bias'], color='steelblue', linewidth=2,
         marker='o', markersize=4, label='ElasticNet bias')
ax3.plot(df['quarter'], combo_metrics['bias'], color='green', linewidth=1.5,
         linestyle=':', marker='^', markersize=4, label='Combination bias')
ax3.axhline(0, color='black', linewidth=0.8)
ax3.axvline(df.loc[16, 'quarter'], color='purple', linewidth=1, linestyle=':', alpha=0.7,
            label='Structural break')
ax3.set_title('Rolling Bias (8Q)', fontweight='bold')
ax3.legend(fontsize=7)
ax3.grid(axis='y', alpha=0.3)
ax3.tick_params(axis='x', rotation=30)

# Panel 4: Model comparison bar chart
model_names = comparison['Model'].tolist()
rmse_vals = comparison['RMSE'].tolist()
bar_colors = ['steelblue', 'green', 'darkorange', 'gray']
bars = ax4.barh(
    model_names, rmse_vals,
    color=bar_colors[:len(model_names)], edgecolor='white', height=0.5
)
for bar, val in zip(bars, rmse_vals):
    ax4.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=9)
ax4.set_title('Model RMSE Comparison', fontweight='bold')
ax4.set_xlabel('RMSE')
ax4.grid(axis='x', alpha=0.3)

fig.suptitle('Nowcasting Model Monitoring Dashboard', fontsize=13, fontweight='bold')
plt.show()
print('Dashboard complete')

## Summary

This notebook demonstrated the complete monitoring workflow:

| Component | Implementation | Key result |
|-----------|---------------|------------|
| Rolling RMSE/MAE/bias | 8-quarter window | Break visible as RMSE spike |
| Bias t-test | Two-sided, $\alpha=0.05$ | Post-break bias detected |
| CUSUM test | Standardised recursive residuals | Break detected within 2Q of true break |
| Re-estimation trigger | Calendar + performance + CUSUM | Calendar fires Q4, performance fires Q18 |
| Model comparison | DM test vs ElasticNet baseline | Combination significantly better (p<0.10) |
| Health report | Structured plain text | Loggable, diffable, auditable |

You now have a complete production monitoring stack. Combine it with the pipeline from Notebook 01 to build a fully operational nowcasting system.